# LHRM — Google Colab Client Inference | Frozen R_AUTHOR Epoch 6

**Client-ready inference notebook — not a training notebook.** Paste an English scientific abstract and get **Top-5 raw AAPD label codes** with sigmoid scores. Works with Colab Free (GPU **recommended**, but CPU inference also supported); **Colab Pro is not required**.

### One-time private Google Drive setup

1. Open [Google Drive](https://drive.google.com/drive/my-drive) on the **same Google account** used with Colab.
2. Create a folder in **My Drive**, named **`LHRM_Client_Inference_Epoch6`**.
3. Upload **exactly these two files** into the folder:
   - `best_dev_model_only.pt` — the original trained checkpoint (**447,952,551 bytes**, ~448 MB) from Kaggle; keep it private.
   - `LHRM_R_AUTHOR_COLAB_CLIENT_SUPPORT_20261015.zip` — accompanying small support package containing the verified R_AUTHOR source and original training evidence.
4. Open this notebook in [Google Colab](https://colab.research.google.com/). You may choose **Runtime → Change runtime type → T4 GPU** (optional) and ensure Internet is available for small tokenizer/dependencies.
5. Run **cells in order, 1 → 6**. Cell 4 verifies byte-exact demo inputs (length and SHA-256) before validating scores; do not edit the built-in replay texts. Approve the **Google Drive permission prompt** in Cell 1. Then paste abstracts in the interactive box in Cell 6.
6. If your folder name/location differs, edit the **`DRIVE_FOLDER`** variable in Cell 1. **Do not** replace it with a public share link.

**Frozen model provenance:** run `AAPD_R_AUTHOR_BCE_RECOVERY_20261015_A`; checkpoint **Epoch 6, Step 40380**, SHA-256 `988bb64fca2b8ea21030585de7268a1cfb1fc8676b91b4d6e6f38c4f22c95643`. The checkpoint includes the fine-tuned BERT backbone — **no full pretrained backbone download** and **no Train/Dev/Test CSV files required**.

**Important scientific caveats:** raw AAPD label-code semantics remain **unverified**. Displayed sigmoid scores are **not calibrated probabilities**. The 0.5 flag is the unchanged training/evaluation display threshold, not a recommendation or adjustment. This demo uses neural CSAM only; **HRRM is disabled** due to no improvement on held-out Dev and was not evaluated on Test. No client inputs are added to any research dataset.

**Privacy:** Colab executes code in a cloud environment. The checkpoint and entered abstracts should be handled according to your organization's data-sharing policy. The notebook does **not** upload your text to Hugging Face; it only downloads public tokenizer/config files and Python dependencies when needed.


In [ ]:
# CELL 1 — CLIENT PREFLIGHT: verify frozen provenance; does not read Train/Dev/Test contents.
from pathlib import Path
from io import BytesIO
import os, sys, json, hashlib, zipfile, shutil, random, time
from google.colab import drive, output
drive.mount("/content/drive")
import numpy as np
import pandas as pd
import torch

RUN_ID = "AAPD_R_AUTHOR_BCE_RECOVERY_20261015_A"
EXPECTED_CHECKPOINT_SHA = '988bb64fca2b8ea21030585de7268a1cfb1fc8676b91b4d6e6f38c4f22c95643'
EXPECTED_EPOCH = 6
EXPECTED_STEP = 40380
EXPECTED_SEL_P1 = 0.8950
EXPECTED_LABEL_ORDER_SHA = "a3d662f3920259a94ed61d83f98f21cabe7f908d98a5662ccd9adc2413cc000b"
EXPECTED_EVIDENCE_ZIP_SHA = "cf27038453034504b37f2fef565c9c0eb47dbf288531797a8a8004c20d426d0e"
EXPECTED_BUNDLE_ZIP_SHA = "e25c583e76d44873e0bdeebcaa99973db8ac4763b79023afd2787902a8e65bbf"
EXPECTED_BUNDLE_MANIFEST_SHA = "8a6ebf146293e22e3a597f992924c84dd06ffc4fe5276d9978add3dfdd9fe9ce"
EXPECTED_EVIDENCE_FILE_SHAS = {'RUN_MANIFEST.json': '75744f07631f6f5977e4ef7941c1d1b8130e24fee389501b188bf18d0a0ab984', 'CONFIG_RESOLVED.json': '72cfcbc306a33c08d3333cebf5760fef15360004207c6c29aeb694dd47712510', 'dev_partition.json': 'fdeea51e0addcd9d52cb577275bf727bb958bb5a0d1bb313bdb9b6f93c5f541b', 'epoch_metrics.csv': 'd0e6116e045252ca075e75757c5ae1bfc6381164fdd431fbb87b87b3abacc122', 'train_history.json': '5af179ee65a4d388b669bfab69650d6a9b766971e8141e7e639e98e7745992f4', 'training_summary.json': '1febc6295962f4cbb837fedf8f92c59f0e898fddeb6cc39c091b52f77bd95f76', 'per_label_metrics.csv': '4c8f89ba4746558011e81d756fd87a76ab2c9ebb451ba2eafbcc3867a98b0067', 'step300_info.json': 'bb2ee0079d3a8c2fcfe6d7245d509eef80649d9bfe9e342388893c0c7cc5aa19', 'hrrm_heldout_comparison.json': 'fadde80fb59a41c54e1e2e80862b2de7ee4299c9d9d3808b4248b2412c14e6c3', 'inference_demo.json': 'a65373c3b08242ba3cc9adfc4f364d5cc0078aaadf0272ad9b874e0810d3b02a', 'neural_dev_predictions.json': '210539710c363c348b061af9387fd0c0f3ebf3ffd43686253e36717cc78aeee1'}
BERT_REPO = "google-bert/bert-base-uncased"
BERT_REVISION = "86b5e0934494bd15c9632b12f734a8a67f723594"
THRESHOLD = 0.5
MAX_LEN = 512
BATCH_SIZE = 8
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Edit this path only if you chose another folder in My Drive.
DRIVE_FOLDER = Path('/content/drive/MyDrive/LHRM_Client_Inference_Epoch6')
WORKING = Path('/content/LHRM_R_AUTHOR_CLIENT_INFERENCE')
WORKING.mkdir(parents=True,exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
def sha256_file(path, block=8*1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(block), b''): h.update(chunk)
    return h.hexdigest()

# Only private Drive files are accepted; no public links and no Kaggle mounts.
assert DRIVE_FOLDER.is_dir(), ('Missing Drive folder: ' + str(DRIVE_FOLDER) +
    '. Create MyDrive/LHRM_Client_Inference_Epoch6 and upload the two specified files.')
CHECKPOINT_PATH = DRIVE_FOLDER / 'best_dev_model_only.pt'
SUPPORT_PATH = DRIVE_FOLDER / 'LHRM_R_AUTHOR_COLAB_CLIENT_SUPPORT_20261015.zip'
assert CHECKPOINT_PATH.is_file(), f'Missing trained checkpoint: {CHECKPOINT_PATH}'
assert SUPPORT_PATH.is_file(), f'Missing support archive: {SUPPORT_PATH}'
assert CHECKPOINT_PATH.stat().st_size == 447952551, 'Wrong checkpoint size — stop.'
assert sha256_file(SUPPORT_PATH) == 'e6f9dec292e49e8eec48c3c47f56d138897953e081b06507f8fa99a615214031', 'Wrong support package SHA — stop.'
with zipfile.ZipFile(SUPPORT_PATH, 'r') as package:
    assert set(package.namelist()) == {'EVIDENCE.zip', 'lhrm_gpu_bundle_r_author_20261015.zip', 'SUPPORT_MANIFEST.json'}
    s_manifest = json.loads(package.read('SUPPORT_MANIFEST.json'))
    assert s_manifest['run_id'] == RUN_ID
    assert s_manifest['files'] == {'EVIDENCE.zip':EXPECTED_EVIDENCE_ZIP_SHA,
          'lhrm_gpu_bundle_r_author_20261015.zip':EXPECTED_BUNDLE_ZIP_SHA}
    raw_evidence_zip = package.read('EVIDENCE.zip')
    raw_bundle_zip = package.read('lhrm_gpu_bundle_r_author_20261015.zip')
assert hashlib.sha256(raw_evidence_zip).hexdigest() == EXPECTED_EVIDENCE_ZIP_SHA
assert hashlib.sha256(raw_bundle_zip).hexdigest() == EXPECTED_BUNDLE_ZIP_SHA
with zipfile.ZipFile(BytesIO(raw_evidence_zip),'r') as evidence_zip:
    EVIDENCE_BYTES = {n:evidence_zip.read(n) for n in EXPECTED_EVIDENCE_FILE_SHAS}
print('[DRIVE] Frozen checkpoint exists:', CHECKPOINT_PATH, '| bytes:', CHECKPOINT_PATH.stat().st_size)
print('[SUPPORT] Source bundle and original evidence integrity: PASS')
for n, expected in EXPECTED_EVIDENCE_FILE_SHAS.items():
    actual = hashlib.sha256(EVIDENCE_BYTES[n]).hexdigest()
    assert actual == expected, f'Evidence hash mismatch: {n}: {actual}'
RUN_MANIFEST = json.loads(EVIDENCE_BYTES['RUN_MANIFEST.json'])
CONFIG = json.loads(EVIDENCE_BYTES['CONFIG_RESOLVED.json'])
assert RUN_MANIFEST['run_id'] == RUN_ID
assert RUN_MANIFEST['best_epoch'] == EXPECTED_EPOCH
assert RUN_MANIFEST['best_step'] == EXPECTED_STEP
assert abs(RUN_MANIFEST['best_selection_p1'] - EXPECTED_SEL_P1) < 1e-10
assert RUN_MANIFEST['label_order_hash'] == EXPECTED_LABEL_ORDER_SHA
assert CONFIG['label_order_hash'] == EXPECTED_LABEL_ORDER_SHA
assert RUN_MANIFEST['selected_checkpoint'] == CHECKPOINT_PATH.name
label_df = pd.read_csv(BytesIO(EVIDENCE_BYTES['per_label_metrics.csv']))
assert len(label_df) == 54 and label_df['index'].tolist() == list(range(54))
LABELS = label_df['label'].tolist()
assert len(set(LABELS)) == 54
assert hashlib.sha256(json.dumps(LABELS).encode()).hexdigest() == EXPECTED_LABEL_ORDER_SHA
print('[EVIDENCE] PASS: run provenance, original label order, 54-label Dev metrics')

# The verified frozen source is supplied entirely inside the support archive.
BUNDLE_BYTES = raw_bundle_zip
assert hashlib.sha256(BUNDLE_BYTES).hexdigest() == EXPECTED_BUNDLE_ZIP_SHA
print('CELL 1 PASS — private Drive artifacts, provenance and 54-label order verified; no dataset splits read.')


In [ ]:
# CELL 2 — Verify R_AUTHOR source and load pinned BERT tokenizer/config (not full pretrained weights).

# Colab: align Transformers API with the verified Kaggle training/replay runtime.
# No attempt to reinstall torch; the checkpoint contains all trained weights.
import subprocess, importlib.metadata
try:
    current_transformers = importlib.metadata.version('transformers')
except importlib.metadata.PackageNotFoundError:
    current_transformers = 'missing'
if current_transformers != '5.16.1':
    print('Installing transformers==5.16.1 for replay compatibility (current:',current_transformers,')')
    subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers==5.16.1','ipywidgets>=8,<9'])
# Imports happen AFTER dependency verification/installation.
from transformers import BertTokenizer, BertConfig, BertModel
import transformers
assert transformers.__version__ == '5.16.1', transformers.__version__

with zipfile.ZipFile(BytesIO(BUNDLE_BYTES),'r') as z:
    raw_manifest = z.read('BUNDLE_MANIFEST.json')
    assert hashlib.sha256(raw_manifest).hexdigest() == EXPECTED_BUNDLE_MANIFEST_SHA
    bundle_manifest = json.loads(raw_manifest)
    source_files = {}
    for rel_path, expected in bundle_manifest['files'].items():
        content = z.read(rel_path)
        assert hashlib.sha256(content).hexdigest() == expected, rel_path
        source_files[rel_path] = content
assert len(source_files) == 16, f'Unexpected bundle file count: {len(source_files)}'
for rel_path, content in source_files.items():
    dest = WORKING/rel_path
    assert dest.resolve().is_relative_to(WORKING.resolve()), 'Unsafe bundle path'
    dest.parent.mkdir(parents=True,exist_ok=True)
    dest.write_bytes(content)
if str(WORKING) not in sys.path: sys.path.insert(0, str(WORKING))
from src.lhrm_baseline.model import LHRMModel
from src.lhrm_baseline.inference import predict_abstract
print('Verified and staged',len(source_files),'R_AUTHOR bundle files.')

# NOTE: best_dev_model_only.pt CONTAINS the fully fine-tuned BERT weights.
# Do NOT download another full pre-trained BERT; the original pinned config+tokenizer suffice.
TOKENIZER_DIR = WORKING/'bert_author_pinned_tokenizer_config'
TOKENIZER_DIR.mkdir(parents=True,exist_ok=True)
required = ['config.json', 'vocab.txt']
if not all((TOKENIZER_DIR/x).exists() for x in required):
    from huggingface_hub import snapshot_download
    try:
        snapshot_download(repo_id=BERT_REPO, revision=BERT_REVISION,
                          allow_patterns=['config.json','vocab.txt','tokenizer_config.json',
                                          'special_tokens_map.json','tokenizer.json'],
                          local_dir=str(TOKENIZER_DIR))
    except Exception as exc:
        raise RuntimeError('Need Colab Internet access for pinned tokenizer/config; ' 
                           'alternatively attach the original official BERT tokenizer/config files.') from exc
assert all((TOKENIZER_DIR/x).exists() for x in required), 'Pinned tokenizer/config files missing.'
bert_config = BertConfig.from_pretrained(str(TOKENIZER_DIR))
assert bert_config.hidden_size == 768 and bert_config.vocab_size == 30522
TOKENIZER = BertTokenizer.from_pretrained(str(TOKENIZER_DIR))
assert len(TOKENIZER) == 30522 and TOKENIZER.pad_token_id == 0
print('CELL 2 PASS — pinned tokenizer + exact verified R_AUTHOR architecture; no dataset splits read.')

In [ ]:
# CELL 3 — Strictly load frozen Epoch-6 trained weights; no optimizer or training.
print('Loading best checkpoint to CPU for provenance checks...')
ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
assert isinstance(ckpt,dict) and isinstance(ckpt.get('model_state_dict'),dict)
assert ckpt.get('run_id') == RUN_ID, f"Unexpected run ID: {ckpt.get('run_id')}"
assert ckpt.get('variant') == 'R_AUTHOR'
assert ckpt.get('epoch') == EXPECTED_EPOCH
assert ckpt.get('step') == EXPECTED_STEP
assert abs(ckpt.get('selection_P@1',-1) - EXPECTED_SEL_P1) < 1e-10
assert ckpt.get('selection_subset') == '400_docs'
assert abs(ckpt.get('full_dev_P@1', -1) - 0.8760) < 1e-10
assert 'optimizer_state_dict' not in ckpt, 'Unexpected training checkpoint, use best_dev_model_only.pt'
sd = ckpt['model_state_dict']
assert tuple(sd['csam.label_emb'].shape) == (54, 768)
assert tuple(sd['csam.classifier.weight'].shape) == (54,1536)
print('Verified frozen checkpoint metadata: Epoch 6, Step 40380, Selection P@1=0.8950.')
assert sha256_file(CHECKPOINT_PATH) == EXPECTED_CHECKPOINT_SHA, 'Wrong checkpoint bytes — STOP'
print('Checkpoint SHA-256 VERIFIED:', EXPECTED_CHECKPOINT_SHA)

# Build identical model architecture. The dummy label_emb is FULLY OVERWRITTEN by trained state_dict.
# This avoids expensive and unnecessary BERT label embedding re-computation.
BERT = BertModel(bert_config)
MODEL = LHRMModel(bert_model=BERT, num_labels=len(LABELS),
                  variant='R_AUTHOR', label_emb=torch.zeros(54,768))
MODEL.load_state_dict(sd, strict=True)
MODEL = MODEL.to(DEVICE).eval()
for p in MODEL.parameters(): p.requires_grad_(False)
assert all(not p.requires_grad for p in MODEL.parameters())
with torch.inference_mode():
    enc = TOKENIZER('Text classification sanity check.', return_tensors='pt',
                    padding='max_length', truncation=True, max_length=MAX_LEN)
    logits, _ = MODEL(enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE))
assert tuple(logits.shape) == (1, 54) and torch.isfinite(logits).all()
print('CELL 3 PASS — verified trained weights loaded strictly, inference-only; no dataset splits read.')

In [ ]:
# CELL 4 — Frozen client inference replay: strict input fingerprint + output verification.
# Texts below are copied BYTE-FOR-BYTE from the original training notebook Cell 7.
# The evidence archive contains only the first 150 characters of each text, so
# the full text SHA-256 checks below are essential for reliable replay.
# FAIL CLOSED: do not enable client inference unless all three inputs and outputs match.
SMOKE_TEXTS = [
    'We present a novel neural architecture for multi-label text classification using contextual similarity attention and label embeddings. Our hierarchical ranking approach learns cross-label dependencies and optimizes ranking metrics directly.',
    'Quantum phase transitions in disordered spin systems are investigated using numerical Monte Carlo simulations. We examine the low-temperature critical exponents and spatial correlation functions across the disordered phase boundary.',
    'Efficient algorithms for convex optimization in high-dimensional machine learning problems. We analyze stochastic gradient methods with adaptive step-sizes and prove linear convergence under strong convexity assumptions.',
]

EXPECTED_SMOKE_INPUTS = [
    ( 240, '66650e1988e209be4c70d9f2ee8b1e4e7c399b1971d30fdd666a4eb57c6640ef' ),
    ( 232, '8c28e37b6a814e44b8f5701bfeb6ee95d6f91c0a13b6d1ea3c4182d35c94054c' ),
    ( 220, '9b148c0704239f7485fb077496dce78f35ce19240b2abd31961a41ea1fd712fe' ),
]

old_demo = json.loads(EVIDENCE_BYTES['inference_demo.json'])
assert len(old_demo) == len(SMOKE_TEXTS) == 3
SMOKE_PASSED = False
for i, (text, reference, fingerprint) in enumerate(zip(SMOKE_TEXTS, old_demo, EXPECTED_SMOKE_INPUTS), 1):
    expected_len, expected_sha = fingerprint
    actual_sha = hashlib.sha256(text.encode('utf-8')).hexdigest()
    assert len(text) == expected_len and actual_sha == expected_sha, (
        f'Smoke {i}: input text changed (length {len(text)} vs {expected_len}); '
        'restore this cell from the verified notebook. DO NOT weaken the score check.'
    )
    expected_preview = text[:150] + ('...' if len(text) > 150 else '')
    assert expected_preview == reference['text_preview'], (
        f'Smoke {i}: evidence preview differs from text; stop and investigate.'
    )
    with torch.inference_mode():
        out = predict_abstract(
            text=text, model=MODEL, tokenizer=TOKENIZER,
            label_order=LABELS, device=DEVICE, hrrm_model=None, top_k=5,
            max_len=MAX_LEN,
        )
    assert out['text_preview'] == expected_preview, f'Smoke {i}: prediction input preview mismatch'
    actual, expected = out['neural_top_k'], reference['neural_top_k']
    assert [x['label'] for x in actual] == [x['label'] for x in expected], f'Smoke {i}: top5 mismatch'
    max_abs_diff = max(abs(a['score'] - e['score']) for a, e in zip(actual, expected))
    assert max_abs_diff < 0.002, f'Smoke {i}: model/tokenizer score mismatch {max_abs_diff}'
    print(f'[{i}/3] PASS | input_sha={actual_sha[:12]} | '
          f'top1={actual[0]["label"]} | max score diff={max_abs_diff:.7f}')
SMOKE_PASSED = True
print('CELL 4 PASS — strict input identity and all three inference replays verified. Ready for new abstracts.')


In [ ]:
# CELL 5 — EDIT THE TEXT BELOW and run this cell to classify ONE new abstract.
# NOTE: Example text is purely a demonstration, NOT ground truth or a validated label mapping.
# Truncation: max 512 BERT tokens; sigmoid scores are NOT calibrated probabilities.

CLIENT_ABSTRACT = """We study representation learning for text classification using transformer
encoders and attention. We compare methods on a collection of scientific abstracts."""
TOP_K = 5  # Show top five ranked codes from the original AAPD dataset.

from IPython.display import display, Markdown

def classify_client_abstract(text, top_k=5, show_table=True):
    if not isinstance(text, str) or not text.strip():
        raise ValueError('Please enter a non-empty English scientific abstract.')
    if len(text) > 100_000:
        raise ValueError('Input too long. Paste a scientific abstract, not a full paper.')
    if not isinstance(top_k, int) or not (1 <= top_k <= len(LABELS)):
        raise ValueError('top_k must be an integer between 1 and 54.')
    if not SMOKE_PASSED:
        raise RuntimeError('Run Cell 4 and verify the original inference replay first.')
    token_count = len(TOKENIZER.encode(text, add_special_tokens=True, truncation=False))
    with torch.inference_mode():
        outcome = predict_abstract(
            text=text.strip(), model=MODEL, tokenizer=TOKENIZER,
            label_order=LABELS, device=DEVICE,
            hrrm_model=None, top_k=top_k, max_len=MAX_LEN,
        )
    rows = [
        {'Rank': int(r['rank']), 'Raw AAPD Label': str(r['label']),
         'Sigmoid score (0–1)': float(r['score']),
         'Above 0.5': 'Yes' if float(r['score']) >= THRESHOLD else 'No'}
        for r in outcome['neural_top_k']
    ]
    result_df = pd.DataFrame(rows)
    if show_table:
        display(Markdown('### Ranked model predictions (not verified topic names)'))
        display(result_df.style.format({'Sigmoid score (0–1)': '{:.4f}'}))
        if token_count > MAX_LEN:
            print(f'WARNING: Input is {token_count} tokens; only first {MAX_LEN} tokens were processed.')
        display(Markdown(
            '**Interpretation:** Original AAPD column codes; mapping of label names to true '
            'topics remains UNVERIFIED. Scores are uncalibrated sigmoid outputs, not factual '
            'confidence guarantees. The 0.5 column is a fixed display cutoff, not tuned for the input.'
        ))
    return {'dataframe': result_df, 'token_count': token_count, 'raw': outcome}

client_result = classify_client_abstract(CLIENT_ABSTRACT, top_k=TOP_K)
print('CELL 5 PASS — client text classified using frozen trained weights (no training).')


In [ ]:
# CELL 6 — Client-friendly interactive text box for Google Colab.
# If widgets do not render, use Cell 5 and edit CLIENT_ABSTRACT instead.

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError:
    print('ipywidgets is unavailable here. Use Cell 5 to paste your abstract and run inference.')
else:
    input_box = widgets.Textarea(
        value='', placeholder='Paste an English scientific abstract here...',
        description='Abstract:', layout=widgets.Layout(width='100%', height='170px'),
        style={'description_width': 'initial'},
    )
    predict_button = widgets.Button(description='Predict Top 5', button_style='primary', icon='search')
    result_box = widgets.Output(layout={'border': '1px solid #ddd', 'padding': '8px'})

    def on_predict_clicked(_):
        with result_box:
            clear_output(wait=True)
            try:
                _ = classify_client_abstract(input_box.value, top_k=5, show_table=True)
            except Exception as exc:
                print('Cannot classify:', type(exc).__name__, str(exc))

    predict_button.on_click(on_predict_clicked)
    display(widgets.VBox([
        widgets.HTML('<h3>LHRM — Client Abstract Classifier</h3><p>'
                     'Frozen Epoch 6 · raw AAPD codes · scores are not calibrated probabilities</p>'),
        input_box, predict_button, result_box,
        widgets.HTML('<small>Warning: semantic label mapping is unresolved. '
                     'This notebook does not train or tune any model.</small>'),
    ]))


## After successful replay

- **Client inference is not a Test evaluation.** For reproducibility, the independent neural **Test** scores of this frozen checkpoint remain: P@1 **0.8490**, Micro-F1 **0.727787**, Macro-F1 **0.555253** across 1,000 documents / 54 labels. Per-label statistics are in the separately supplied final report.
- **No AAPD CSV files are opened** by this notebook. No training, tuning, thresholds changes, or HRRM re-ranking are performed.
- Only `best_dev_model_only.pt` and the support ZIP are needed in your **private** Google Drive. The support ZIP intentionally excludes trained weights; the full checkpoint remains a separate, larger private artifact.
- If Google Drive access is disconnected, remount and rerun cells **1 → 4**, then use Cells 5/6. Do not silently skip failed checkpoint/hash/replay validations.
- A successful output table proves the model runs; it does **not** establish that label names correspond to scientifically correct subjects. The raw AAPD label-code mapping is still unverified.
- When sharing with another client account, explicitly grant approved access to the private checkpoint and support files; **do not** publish them or hard-code any tokens/share links into this notebook.
